In [ ]:
import pandas as pd

df = pd.read_csv('/content/movies.csv')
df.head()

In [ ]:
import pandas as pd

df = pd.read_csv('/content/movies.csv', low_memory=False)
df.head()

In [ ]:
print(df.info())
print(df.isnull().sum())

In [ ]:
import numpy as np
import json

# 1. Convert 'release_date' to datetime and handle errors
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# 2. Create 'release_year' column
df['release_year'] = df['release_date'].dt.year

# 3. Convert 'budget' and 'revenue' to numeric, coercing errors
df['budget'] = pd.to_numeric(df['budget'], errors='coerce').fillna(0)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)

# 4. Convert 'runtime' to numeric, coercing errors
df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')

# 5. Process the 'genres' column
def parse_genres(genres_str):
    if pd.isna(genres_str) or genres_str == '[]':
        return []
    try:
        genres_list = json.loads(genres_str)
        return [genre['name'] for genre in genres_list]
    except (json.JSONDecodeError, TypeError):
        return []

df['genres'] = df['genres'].apply(parse_genres)

# 6. Display the first few rows of the DataFrame with the cleaned and processed columns
df.head()

In [ ]:
import ast # Import the ast module for literal_eval

# Refined function to process the 'genres' column using ast.literal_eval
def parse_genres_corrected(genres_str):
    if pd.isna(genres_str) or genres_str == '[]':
        return []
    try:
        # Use ast.literal_eval for safer parsing of Python literal structures
        genres_list = ast.literal_eval(genres_str)
        # Ensure the parsed result is a list before list comprehension
        if isinstance(genres_list, list):
            return [genre['name'] for genre in genres_list if isinstance(genre, dict) and 'name' in genre]
        return []
    except (ValueError, SyntaxError, TypeError):
        return []

df['genres'] = df['genres'].apply(parse_genres_corrected)

# Display the first few rows of the DataFrame with the corrected 'genres' column
df.head()

In [ ]:
import ast # Import the ast module for literal_eval

# Refined function to process the 'genres' column using ast.literal_eval
def parse_genres_corrected(genres_str):
    if pd.isna(genres_str):
        return []
    try:
        # Use ast.literal_eval for safer parsing of Python literal structures
        # This will also correctly parse the string '[]' into an empty list
        genres_list = ast.literal_eval(genres_str)
        # Ensure the parsed result is a list before list comprehension
        if isinstance(genres_list, list):
            return [genre['name'] for genre in genres_list if isinstance(genre, dict) and 'name' in genre]
        return [] # Return empty if it was parsed but not a list (e.g., 'None')
    except (ValueError, SyntaxError, TypeError):
        # Catch errors if it's not a valid Python literal or malformed
        return []

df['genres'] = df['genres'].apply(parse_genres_corrected)

# Display the first few rows of the DataFrame with the corrected 'genres' column
df.head()

In [ ]:
import ast # Import the ast module for literal_eval

# Refined function to process the 'genres' column using ast.literal_eval
def parse_genres_corrected(genres_str):
    try:
        # Use ast.literal_eval for safer parsing of Python literal structures
        # This will also correctly parse the string '[]' into an empty list.
        # If genres_str is not a string (e.g., NaN, None), ast.literal_eval will raise a TypeError,
        # which is handled by the except block.
        genres_list = ast.literal_eval(genres_str)

        # Ensure the parsed result is a list before list comprehension
        if isinstance(genres_list, list):
            return [genre['name'] for genre in genres_list if isinstance(genre, dict) and 'name' in genre]
        return [] # Return empty if it was parsed but not a list (e.g., 'None' string becomes None)
    except (ValueError, SyntaxError, TypeError):
        # Catch errors if it's not a valid Python literal or malformed (e.g., NaN, None, malformed strings)
        return []

df['genres'] = df['genres'].apply(parse_genres_corrected)

# Display the first few rows of the DataFrame with the corrected 'genres' column
df.head()

In [ ]:
movies_per_year = df['release_year'].value_counts().sort_index()
print("Number of movies released per year:")
print(movies_per_year.head())

In [ ]:
genre_counts = df.explode('genres')['genres'].value_counts()
print("Popularity of genres:")
print(genre_counts.head())

In [ ]:
import ast # Ensure ast is imported

# Refined function to process the 'genres' column using ast.literal_eval
def parse_genres_corrected(genres_str):
    try:
        # ast.literal_eval expects a string. Convert to string first if it isn't.
        # This handles cases where genres_str might be NaN (float) or NoneType.
        if not isinstance(genres_str, str):
            genres_str = str(genres_str)

        # If after converting to string, it's 'nan' or an empty string from some conversion,
        # treat it as an empty list to avoid ValueError from ast.literal_eval.
        if genres_str.strip() in ['nan', '']:
            return []

        genres_list = ast.literal_eval(genres_str)

        if isinstance(genres_list, list):
            return [genre['name'] for genre in genres_list if isinstance(genre, dict) and 'name' in genre]
        return []
    except (ValueError, SyntaxError, TypeError):
        return []

# Apply the parsing function to the 'genres' column
df['genres'] = df['genres'].apply(parse_genres_corrected)

# Display the first few rows of the DataFrame with the corrected 'genres' column
df.head()

In [ ]:
import ast # Ensure ast is imported

# Reload the original 'genres' column from the CSV to ensure we start from raw string data
original_genres_series = pd.read_csv('/content/movies.csv', low_memory=False)['genres']

# Refined function to process the 'genres' column using ast.literal_eval
def parse_genres_corrected(genres_str):
    try:
        # Ensure input is a string for ast.literal_eval
        if not isinstance(genres_str, str):
            genres_str = str(genres_str)

        # Handle 'nan' strings or empty strings that may result from conversion
        if genres_str.strip() in ['nan', '']:
            return []

        genres_list = ast.literal_eval(genres_str)

        if isinstance(genres_list, list):
            return [genre['name'] for genre in genres_list if isinstance(genre, dict) and 'name' in genre]
        return [] # Return empty if parsed but not a list
    except (ValueError, SyntaxError, TypeError):
        return []

# Apply the parsing function to the freshly loaded 'genres' series
df['genres'] = original_genres_series.apply(parse_genres_corrected)

# Display the first few rows of the DataFrame with the corrected 'genres' column
df.head()

In [ ]:
genre_counts = df.explode('genres')['genres'].value_counts()
print("Popularity of genres:")
print(genre_counts.head())

In [ ]:
genre_avg_rating = df.explode('genres').groupby('genres')['vote_average'].mean().sort_values(ascending=False)
print("Average rating per genre:")
print(genre_avg_rating.head())

In [ ]:
genre_avg_runtime = df.explode('genres').groupby('genres')['runtime'].mean().sort_values(ascending=False)
print("Average runtime per genre:")
print(genre_avg_runtime.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better aesthetics
sns.set_style("whitegrid")

# 1. Line plot for movie release trends over the years
plt.figure(figsize=(12, 6))
plt.plot(movies_per_year.index, movies_per_year.values, marker='o', linestyle='-')
plt.xlabel("Release Year")
plt.ylabel("Number of Movies Released")
plt.title("Movie Release Trends Over Years")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Bar chart for top 10 most popular genres
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_counts.head(10).index, y=genre_counts.head(10).values, palette='viridis')
plt.xlabel("Genre")
plt.ylabel("Number of Movies")
plt.title("Top 10 Most Popular Genres")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 3. Bar chart for top 10 genres by average rating
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_avg_rating.head(10).index, y=genre_avg_rating.head(10).values, palette='plasma')
plt.xlabel("Genre")
plt.ylabel("Average Rating")
plt.title("Top 10 Genres by Average Rating")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Bar chart for top 10 genres by average runtime
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_avg_runtime.head(10).index, y=genre_avg_runtime.head(10).values, palette='magma')
plt.xlabel("Genre")
plt.ylabel("Average Runtime (minutes)")
plt.title("Top 10 Genres by Average Runtime")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better aesthetics
sns.set_style("whitegrid")

# 1. Line plot for movie release trends over the years
plt.figure(figsize=(12, 6))
plt.plot(movies_per_year.index, movies_per_year.values, marker='o', linestyle='-')
plt.xlabel("Release Year")
plt.ylabel("Number of Movies Released")
plt.title("Movie Release Trends Over Years")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Bar chart for top 10 most popular genres
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_counts.head(10).index, y=genre_counts.head(10).values, hue=genre_counts.head(10).index, palette='viridis', legend=False)
plt.xlabel("Genre")
plt.ylabel("Number of Movies")
plt.title("Top 10 Most Popular Genres")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 3. Bar chart for top 10 genres by average rating
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_avg_rating.head(10).index, y=genre_avg_rating.head(10).values, hue=genre_avg_rating.head(10).index, palette='plasma', legend=False)
plt.xlabel("Genre")
plt.ylabel("Average Rating")
plt.title("Top 10 Genres by Average Rating")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 4. Bar chart for top 10 genres by average runtime
plt.figure(figsize=(12, 7))
sns.barplot(x=genre_avg_runtime.head(10).index, y=genre_avg_runtime.head(10).values, hue=genre_avg_runtime.head(10).index, palette='magma', legend=False)
plt.xlabel("Genre")
plt.ylabel("Average Runtime (minutes)")
plt.title("Top 10 Genres by Average Runtime")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
from pyspark.sql import SparkSession

# Create a SparkSession instance
spark = SparkSession.builder.appName("MovieAnalysis").getOrCreate()

# Load the movies.csv file into a Spark DataFrame
# Ensure that headers are included and the schema is inferred
spark_df = spark.read.csv('/content/movies.csv', header=True, inferSchema=True)

# Display the schema of spark_df
print("Schema of spark_df:")
spark_df.printSchema()

# Show the first 5 rows of spark_df
print("First 5 rows of spark_df:")
spark_df.show(5)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 2. Print the current schema of the spark_df to verify data types.
print("Current Schema of spark_df:")
spark_df.printSchema()

# 3. Calculate the count of null values for each column
print("\nNull values count for each column:")
null_counts = spark_df.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in spark_df.columns])
null_counts.show()

# 4. Convert 'release_date' to a DateType and extract the year
spark_df = spark_df.withColumn('release_date', F.to_date(F.col('release_date'), 'yyyy-MM-dd')) \
                   .withColumn('release_year', F.year(F.col('release_date')))

# 5. Convert numeric columns, handling non-numeric values and filling nulls
spark_df = spark_df.withColumn('budget', F.coalesce(F.col('budget').cast(T.LongType()), F.lit(0))) \
                   .withColumn('revenue', F.coalesce(F.col('revenue').cast(T.LongType()), F.lit(0))) \
                   .withColumn('runtime', F.col('runtime').cast(T.DoubleType())) \
                   .withColumn('popularity', F.col('popularity').cast(T.DoubleType())) \
                   .withColumn('vote_average', F.col('vote_average').cast(T.DoubleType())) \
                   .withColumn('vote_count', F.col('vote_count').cast(T.DoubleType())) \
                   .withColumn('id', F.col('id').cast(T.IntegerType()))

# 6. Convert 'adult' and 'video' columns to BooleanType
spark_df = spark_df.withColumn('adult', F.col('adult').cast(T.BooleanType())) \
                   .withColumn('video', F.col('video').cast(T.BooleanType()))

# 7. Define a schema for the JSON structure within the 'genres' column
genre_schema = T.ArrayType(T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True)
]))

# 8. Use F.from_json to parse the 'genres' column and extract genre names
# First, cast genres to StringType to ensure from_json can process it
# Then, parse it, and use transform to get just the 'name'
spark_df = spark_df.withColumn("genres",
    F.when(F.col("genres").isNotNull(), F.from_json(F.col("genres").cast(T.StringType()), genre_schema))
     .otherwise(F.array().cast(genre_schema))
)
spark_df = spark_df.withColumn("genres", F.transform(F.col("genres"), lambda x: x.getItem("name")))


# 9. Display the updated schema of spark_df
print("\nUpdated Schema of spark_df:")
spark_df.printSchema()

# 10. Show the first 5 rows of the cleaned spark_df
print("\nFirst 5 rows of cleaned spark_df:")
spark_df.show(5, truncate=False)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1. Stop the current SparkSession to ensure a clean state.
spark.stop()

# 2. Create a new SparkSession instance
spark = SparkSession.builder.appName("MovieAnalysis_Retry").getOrCreate()

# 3. Define a custom_schema where all columns are initially read as StringType.
# This prevents Spark from inferring incorrect types, especially for mixed-type columns.
custom_schema = T.StructType([
    T.StructField("adult", T.StringType(), True),
    T.StructField("belongs_to_collection", T.StringType(), True),
    T.StructField("budget", T.StringType(), True),
    T.StructField("genres", T.StringType(), True),
    T.StructField("homepage", T.StringType(), True),
    T.StructField("id", T.StringType(), True),
    T.StructField("imdb_id", T.StringType(), True),
    T.StructField("original_language", T.StringType(), True),
    T.StructField("original_title", T.StringType(), True),
    T.StructField("overview", T.StringType(), True),
    T.StructField("popularity", T.StringType(), True),
    T.StructField("poster_path", T.StringType(), True),
    T.StructField("production_companies", T.StringType(), True),
    T.StructField("production_countries", T.StringType(), True),
    T.StructField("release_date", T.StringType(), True),
    T.StructField("revenue", T.StringType(), True),
    T.StructField("runtime", T.StringType(), True),
    T.StructField("spoken_languages", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("tagline", T.StringType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("video", T.StringType(), True),
    T.StructField("vote_average", T.StringType(), True),
    T.StructField("vote_count", T.StringType(), True)
])

# 4. Load the movies.csv file into a Spark DataFrame using this custom_schema
spark_df = spark.read.csv(
    '/content/movies.csv',
    header=True,
    schema=custom_schema, # Use the predefined schema
    multiLine=True,       # Handle potential newlines within fields
    escape='"'          # Handle escaped quotes within fields
)

# 5. Print the schema of the newly loaded Spark DataFrame
print("Schema of spark_df after custom schema load:")
spark_df.printSchema()

# 6. Display the first 5 rows of the DataFrame to verify the initial loading
print("First 5 rows of spark_df after custom schema load:")
spark_df.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 7. Convert the 'release_date' column to a DateType using F.expr("try_to_timestamp(release_date, 'yyyy-MM-dd')").cast(T.DateType())
#    to gracefully handle malformed date strings, and then extract the 'release_year'.
spark_df = spark_df.withColumn('release_date', F.expr("try_to_timestamp(release_date, 'yyyy-MM-dd')").cast(T.DateType())) \
                   .withColumn('release_year', F.year(F.col('release_date')))

# 8. Convert 'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count', and 'id' columns to their appropriate numeric types
#    (LongType, DoubleType, IntegerType) using .cast() and F.coalesce() to fill NULL values with 0 where appropriate,
#    allowing Spark to handle non-numeric values by returning NULL.
spark_df = spark_df.withColumn('budget', F.coalesce(F.col('budget').cast(T.LongType()), F.lit(0))) \
                   .withColumn('revenue', F.coalesce(F.col('revenue').cast(T.LongType()), F.lit(0))) \
                   .withColumn('runtime', F.col('runtime').cast(T.DoubleType())) \
                   .withColumn('popularity', F.col('popularity').cast(T.DoubleType())) \
                   .withColumn('vote_average', F.col('vote_average').cast(T.DoubleType())) \
                   .withColumn('vote_count', F.col('vote_count').cast(T.DoubleType())) \
                   .withColumn('id', F.col('id').cast(T.IntegerType()))

# 9. Convert 'adult' and 'video' columns to BooleanType using .cast(T.BooleanType()).
spark_df = spark_df.withColumn('adult', F.col('adult').cast(T.BooleanType())) \
                   .withColumn('video', F.col('video').cast(T.BooleanType()))


# 10. Define a schema for the JSON structure within the 'genres' column, then use F.from_json and F.transform to parse
#     the 'genres' column and extract genre names into an array of strings. Handle potential NULL or invalid JSON strings by returning an empty array.
genre_schema = T.ArrayType(T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True)
]))

# Use F.from_json to parse the 'genres' column and extract genre names
# Handle cases where 'genres' column might not be a valid JSON string by using F.when().otherwise(F.array())
spark_df = spark_df.withColumn("genres",
    F.when(
        F.col("genres").isNotNull() & (F.length(F.trim(F.col("genres"))) > 0) & (F.col("genres") != "[]"),
        F.from_json(F.col("genres"), genre_schema)
    ).otherwise(F.array().cast(genre_schema))
)
spark_df = spark_df.withColumn("genres", F.transform(F.col("genres"), lambda x: x.getItem("name")))

# 11. Print the updated schema of the cleaned Spark DataFrame.
print("\nUpdated Schema of spark_df:")
spark_df.printSchema()

# 12. Display the first 5 rows of the fully cleaned Spark DataFrame to confirm all transformations.
print("\nFirst 5 rows of cleaned spark_df:")
spark_df.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Ensure SparkSession is available (it was created in previous steps)
# If it wasn't, you would re-create it like: spark = SparkSession.builder.appName("WordCount").getOrCreate()

# Load the text file into a Spark DataFrame
text_df = spark.read.text("/content/big.txt")

# Tokenize the lines into words, convert to lowercase, and remove non-alphanumeric characters
# Then flatten the array of words using explode
words_df = text_df.select(F.explode(F.split(F.lower(F.regexp_replace("value", "[^a-zA-Z\\s]", "")), "\\s+")).alias("word"))

# Filter out empty strings that might result from splitting or cleaning
words_df = words_df.filter(F.col("word") != "")

# Perform word count
word_counts = words_df.groupBy("word").count()

# Order by count in descending order and show the top 10 words
print("\nTop 10 most frequent words from /content/big.txt:")
word_counts.orderBy(F.desc("count")).show(10)

In [ ]:
from pyspark.sql import functions as F

# Register the cleaned Spark DataFrame as a temporary SQL view
spark_df.createOrReplaceTempView("movies")

# Query for movies released in 1993 using Spark SQL
movies_1993_sql = spark.sql("SELECT title, release_year FROM movies WHERE release_year = 1993")

# Show the results
print("Movies released in 1993 (using Spark SQL):")
movies_1993_sql.show(10, truncate=False)
